# Numerical Methods — Interpolation & Linear Systems  
---
**Name: Manmath Maroti Kornule**   
**PRN: 202401110045**
---
---    

This notebook includes implementations of:

1. Lagrange Interpolation  
2. Newton’s Divided Difference Interpolation  
3. LU Decomposition for solving linear systems  

All solutions include clean explanations and example runs.


# 1. Lagrange's Interpolation Method

Given data points:

$$
(x_0, y_0), (x_1, y_1), \ldots, (x_n, y_n)
$$

The Lagrange polynomial is:

$$
P(x) = \sum_{i=0}^n y_i \, L_i(x)
$$

where each basis polynomial is:

$$
L_i(x) = \prod_{\substack{j=0, j \ne i}}^{n}
\frac{x - x_j}{x_i - x_j}
$$

This method constructs a polynomial that passes through all given data points.


In [1]:
def lagrange_interpolation(x_values, y_values, x):
    total = 0
    n = len(x_values)

    for i in range(n):
        term = y_values[i]
        for j in range(n):
            if i != j:
                term *= (x - x_values[j]) / (x_values[i] - x_values[j])
        total += term

    return total

# Example
x_vals = [0, 1, 2]
y_vals = [1, 3, 2]

lagrange_interpolation(x_vals, y_vals, 1.5)


2.875

# 2. Newton’s Divided Difference Interpolation

The divided difference table starts with:

$$
f[x_i] = y_i
$$

First-order differences:

$$
f[x_i, x_{i+1}] =
\frac{f[x_{i+1}] - f[x_i]}{x_{i+1} - x_i}
$$

Higher-order differences:

$$
f[x_i, \ldots, x_{i+k}] =
\frac{f[x_{i+1}, \ldots, x_{i+k}] - f[x_i, \ldots, x_{i+k-1}]}
{x_{i+k} - x_i}
$$

Newton interpolation polynomial:

$$
P(x) = a_0
+ a_1(x - x_0)
+ a_2(x - x_0)(x - x_1)
+ \cdots
$$

Where $a_i$ are the first-row elements of the divided difference table.


In [2]:
import numpy as np

def newton_divided_diff(x, y):
    n = len(x)
    table = np.zeros((n, n))
    table[:, 0] = y

    for j in range(1, n):
        for i in range(n - j):
            table[i][j] = (table[i+1][j-1] - table[i][j-1]) / (x[i+j] - x[i])

    return table

def newton_interpolation(x, y, value):
    table = newton_divided_diff(x, y)
    n = len(x)
    result = table[0][0]
    product = 1

    for i in range(1, n):
        product *= (value - x[i-1])
        result += table[0][i] * product

    return result

# Example
x_vals = [0, 1, 2]
y_vals = [1, 3, 2]

newton_interpolation(x_vals, y_vals, 1.5)


np.float64(2.875)

# 3. LU Decomposition for Solving Linear Systems

We solve the system:

$$
AX = B
$$

Using matrix factorization:

$$
A = LU
$$

Where:

- $L$: Lower triangular matrix  
- $U$: Upper triangular matrix  

Then solve two simpler systems:

1. Forward substitution:

$$
LY = B
$$

2. Backward substitution:

$$
UX = Y
$$

This method is faster and more stable for solving multiple systems with the same matrix $A$.


In [3]:
def lu_decomposition(A):
    n = len(A)
    L = np.zeros((n,n))
    U = np.zeros((n,n))

    for i in range(n):
        # Upper triangular
        for k in range(i, n):
            sum_val = sum(L[i][j] * U[j][k] for j in range(i))
            U[i][k] = A[i][k] - sum_val

        # Lower triangular
        for k in range(i, n):
            if i == k:
                L[i][i] = 1
            else:
                sum_val = sum(L[k][j] * U[j][i] for j in range(i))
                L[k][i] = (A[k][i] - sum_val) / U[i][i]

    return L, U

def solve_lu(A, B):
    L, U = lu_decomposition(A)
    n = len(A)

    # Forward substitution (Ly = B)
    Y = np.zeros(n)
    for i in range(n):
        Y[i] = B[i] - sum(L[i][j] * Y[j] for j in range(i))

    # Backward substitution (Ux = Y)
    X = np.zeros(n)
    for i in reversed(range(n)):
        X[i] = (Y[i] - sum(U[i][j] * X[j] for j in range(i+1, n))) / U[i][i]

    return X, L, U

# Example:
A = np.array([[2, -1, 1],
              [3, 3, 9],
              [3, 3, 5]], dtype=float)

B = np.array([8, 0, 6], dtype=float)

solution, L, U = solve_lu(A, B)
solution


array([ 4.66666667, -0.16666667, -1.5       ])